# MiniMax H3 — ComfyUI on Colab with private Tailscale access

Use Colab's GPU from the ComfyUI interface on your PC. Keep Tailscale connected on your PC.

1. Install ComfyUI and H3 custom nodes; no model weights yet.
2. In Section 4, install the small Tailscale client and sign in using the same account as your PC.
3. Section 5 starts/reuses ComfyUI and prints the private address for this runtime.
4. Open that address from your PC and verify the canvas, menus and live connection.
5. Only then approve and download the original H3 model stack.
6. Restart only ComfyUI; Tailscale stays connected.

**No Google Drive mount or persistence.** Download wanted outputs and workflow JSON directly from ComfyUI before deleting the runtime. A fresh runtime needs a new Tailscale sign-in and local model downloads.

**Already installed and running in this runtime?** Run Sections 0, 2, 4 and 5 to add the private route. Skip reinstalling and downloading models you already have locally. Existing Cloudflare processes are left alone during this migration; the private address printed here uses Tailscale.

This notebook never queues generation for you. Local health checks do not prove browser or GPU generation success.


## 0. Local runtime storage

All models, inputs, outputs and workflow state stay on the Colab VM. Download results directly from the ComfyUI workflow before ending the runtime. No Google Drive mount or synchronization is used.

In [ ]:
import os
from pathlib import Path

os.environ['H3_ACCESS_METHOD'] = 'tailscale'

COMFY_ROOT = '/content/ComfyUI'
MODEL_ROOT = Path(COMFY_ROOT) / 'models'
print('Local ComfyUI storage:', COMFY_ROOT)
print('Save results using Download in the ComfyUI workflow before ending this runtime.')


## 1. Check GPU + safe runtime settings

In [ ]:
import subprocess, os

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader | head -n1')
vram_mb = int(sh('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1'))
vram_gb = vram_mb / 1024
name = gpu_name.lower()

if 'rtx pro 6000' in name or 'blackwell' in name:
    H3_RESERVE_VRAM_GB = '6'
elif 'a100' in name:
    H3_RESERVE_VRAM_GB = '4'
elif 'l4' in name:
    H3_RESERVE_VRAM_GB = '2'
else:
    H3_RESERVE_VRAM_GB = '1'

print(f'GPU: {gpu_name} ({vram_gb:.1f} GB)')
print('Reserved VRAM:', H3_RESERVE_VRAM_GB, 'GB')


## 2. Clone/update the H3 branch

In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 3. Install/update ComfyUI + H3 custom nodes — NO model weights yet

This installs ComfyUI and the custom-node code needed by your H3/Director workflows. It does **not** download the large H3 model weights.

Installer output appears live below and is appended to `/content/h3_comfy_logs/setup.log`. A failure reports its stage and shell line. Installation must succeed before preflight or any model download.

In [ ]:
import os, subprocess, sys

os.environ['H3_ACCESS_METHOD'] = 'tailscale'
os.environ['COMFY_ROOT'] = '/content/ComfyUI'
os.environ['H3_VRAM_MODE'] = 'auto'
os.environ['H3_RESERVE_VRAM_GB'] = H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD'] = 'none'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

sys.path.insert(0, '/content/minimax_h3_comfy')
import importlib, comfy_preflight
importlib.reload(comfy_preflight)  # Pick up fixes after rerunning the branch-fetch cell.
run_setup_command = comfy_preflight.run_setup_command

# Stream both stdout and stderr into this cell; append a redacted setup log.
run_setup_command(['bash', '/content/minimax_h3_comfy/install_comfy_h3.sh'],
                  label='ComfyUI + H3 installation')

CUSTOM='/content/ComfyUI/custom_nodes'
def clone_or_pull(url, folder):
    path=f'{CUSTOM}/{folder}'
    if os.path.isdir(path+'/.git'):
        run_setup_command(['git','-C',path,'pull','--ff-only'], label=folder + ' update')
    else:
        run_setup_command(['git','clone','--depth','1',url,path], label=folder + ' clone')
    req=os.path.join(path,'requirements.txt')
    if os.path.exists(req):
        run_setup_command([sys.executable,'-m','pip','install','-r',req], label=folder + ' requirements')

clone_or_pull('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-VideoHelperSuite')
clone_or_pull('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes')
clone_or_pull('https://github.com/pixaroma/ComfyUI-Pixaroma.git','ComfyUI-Pixaroma')
run_setup_command([sys.executable,'-m','pip','install','-U','huggingface_hub'], label='Hugging Face Hub package')

print('✅ ComfyUI + H3/Director custom nodes installed.')
print('✅ No H3 model weights have been downloaded by this notebook yet.')


## 4. Connect this Colab runtime to Tailscale

Run this cell, open the sign-in link it prints, and use **the same Tailscale account as your PC**. Approve the `colab-comfy` device if asked, then continue to Section 5. If your network requires administrator device approval, finish that first.

No Cloudflare token or Tailscale auth key is needed. The client uses userspace networking and keeps its identity only in runtime memory. Rerunning this cell reuses the managed daemon and existing login. Authentication links are shown here but redacted from saved diagnostic logs; do not share notebook outputs containing a pending login link.


In [ ]:
import os, sys, importlib
from urllib.parse import urlparse
from IPython.display import display, Markdown

H3_PREFLIGHT_COMPLETE = False
PROCEED_WITH_H3_MODEL_DOWNLOADS = False
os.environ['H3_ACCESS_METHOD'] = 'tailscale'
sys.path.insert(0, '/content/minimax_h3_comfy')
import comfy_preflight
importlib.reload(comfy_preflight)
comfy_preflight.run_launcher('tailscale-login')
_ts_status = comfy_preflight.tailscale_status()
if _ts_status['BackendState'] == 'Running':
    print('Tailscale connected. Continue to Section 5.')
elif _ts_status['BackendState'] == 'NeedsMachineAuth':
    print('Approve colab-comfy in your Tailscale admin console, then run Section 5.')
else:
    _login_url = _ts_status.get('AuthURL', '')
    _parsed = urlparse(_login_url)
    if _parsed.scheme != 'https' or _parsed.netloc != 'login.tailscale.com':
        raise SystemExit('No valid sign-in link. Rerun Section 4 and inspect diagnostics.')
    display(Markdown(f'[Sign this Colab runtime into Tailscale]({_login_url})'))
    print('Finish sign-in using the same account as your PC, then run Section 5.')
del _ts_status


## 5. PRE-FLIGHT — Start ComfyUI + private Tailscale access

Complete Section 4 sign-in first. This cell prints a private `http://100.x.x.x:8188/` address. Keep Tailscale connected on your PC and paste the printed address directly into the browser address bar.

Verify the full ComfyUI canvas and menus load and the connection stays active. Empty model dropdowns are expected before the approved downloads. The private route is encrypted by Tailscale and available to devices permitted by your Tailscale network policy, not the public internet. No public Funnel is enabled.


In [ ]:
import os, sys, importlib
os.environ['H3_ACCESS_METHOD'] = 'tailscale'

H3_PREFLIGHT_COMPLETE = False
H3_MODELS_DOWNLOADED = False
PROCEED_WITH_H3_MODEL_DOWNLOADS = False
sys.path.insert(0, '/content/minimax_h3_comfy')
import comfy_preflight
importlib.reload(comfy_preflight)
comfy_preflight.run_launcher('preflight')
COMFY_URL = comfy_preflight.access_url()
print('Open on your PC:', COMFY_URL)
H3_PREFLIGHT_COMPLETE = True
print('Browser verification is still required; no model download is approved yet.')


## 6. HARD STOP / approval gate

The default is `False`, so **Run all stops here before downloading any H3 weights**.

First open the private Tailscale address printed in Section 5. If the full ComfyUI UI works, change the value below to `True` and run this cell again, then continue to Section 7.

In [ ]:
PROCEED_WITH_H3_MODEL_DOWNLOADS = False

if PROCEED_WITH_H3_MODEL_DOWNLOADS is not True:
    print('Verify the private ComfyUI address from Section 5 first.\n'
          'If the full ComfyUI UI loads, set\n'
          'PROCEED_WITH_H3_MODEL_DOWNLOADS = True\n'
          'and continue.')
    raise SystemExit('Paused before model downloads. ComfyUI + Tailscale remain running.')

if globals().get('H3_PREFLIGHT_COMPLETE') is not True:
    raise SystemExit('Run the ComfyUI + Tailscale preflight cell first.')
print('Approved. Continue to Section 7.')


## 7. Download the H3 model stack — only after approval

In [ ]:
# Guard this cell too: running it directly must never bypass approval.
if (globals().get('PROCEED_WITH_H3_MODEL_DOWNLOADS') is not True
        or globals().get('H3_PREFLIGHT_COMPLETE') is not True):
    raise SystemExit('Verify the private ComfyUI address from Section 5 first, then approve in Section 6.')
H3_MODELS_DOWNLOADED = False
import subprocess
comfy_preflight.run_launcher('check')

from huggingface_hub import hf_hub_download
from pathlib import Path

MODEL_ROOT = Path('/content/ComfyUI/models')
for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models']:
    (MODEL_ROOT/folder).mkdir(parents=True, exist_ok=True)

downloads = [
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_ref2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', MODEL_ROOT, MODEL_ROOT/'text_encoders'/'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_video_vae_fp16.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_video_vae_fp16.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_audio_vae_fp32.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_audio_vae_fp32.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'),
    ('LBH-123-AI/Minimax_h3_latent_Upscaler','minimax_h3_latent_upscaler_3d_fp16.safetensors', MODEL_ROOT/'latent_upscale_models', MODEL_ROOT/'latent_upscale_models'/'minimax_h3_latent_upscaler_3d_fp16.safetensors')
]

for repo_id, filename, local_dir, target in downloads:
    if target.exists() and target.stat().st_size > 1024*1024:
        print('SKIP', target.name)
        continue
    print('DOWNLOAD', filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(local_dir))
    if not target.exists():
        raise FileNotFoundError(f'Expected model was not created: {target}')
    print('READY', target.name)

print('✅ H3 model stack ready at:', MODEL_ROOT)

H3_MODELS_DOWNLOADED = True


## 8. Restart only ComfyUI so the new models appear

Tailscale stays running. Refresh the same private address after this cell succeeds.


In [ ]:
if (globals().get('PROCEED_WITH_H3_MODEL_DOWNLOADS') is not True
        or globals().get('H3_MODELS_DOWNLOADED') is not True):
    raise SystemExit('Complete the approved model-download cell before the final restart.')
import subprocess
comfy_preflight.run_launcher('restart')


## Diagnostics — safe before model downloads

Run this cell at any time after Section 2, including after the intentional stop. It reports the local HTTP status, listener, connector PIDs and Tailscale login state and the last 50 lines of each log, with tokens/credentials redacted. It does not start services or download models.

Local health and tunnel registration do not prove browser UI/WebSocket success. In your browser, confirm the real ComfyUI canvas and menus load and the connection stays active before approving. After generation, download/save your outputs before disconnecting and deleting the Colab runtime; you control both steps.

In [ ]:
import os, sys, importlib
os.environ['H3_ACCESS_METHOD'] = 'tailscale'
sys.path.insert(0, '/content/minimax_h3_comfy')
import comfy_preflight
importlib.reload(comfy_preflight)
comfy_preflight.diagnostics()
